In [4]:
import os, shutil, json, zipfile

INPUT = "/kaggle/input/datasets/gauravvybhavkabhai/math500-7b-adapter"  # your dataset name

# Copy adapter folder
shutil.copytree(os.path.join(INPUT, "mathloramixed"), "/kaggle/working/mathloramixed")
print("Adapter copied")

# Copy checkpoint, remove broken LoRAAdaptive
with open(os.path.join(INPUT, "mathcheckpoint.json")) as f:
    ckpt = json.load(f)
ckpt["results"] = [r for r in ckpt["results"] if r["Method"] != "LoRAAdaptive"]
with open("/kaggle/working/mathcheckpoint.json", "w") as f:
    json.dump(ckpt, f, indent=2)
print(f"Checkpoint ready: {len(ckpt['results'])} methods")

#/mathloramixed

Adapter copied
Checkpoint ready: 30 methods


In [1]:
#!/usr/bin/env python3
"""
TokenSkip End-to-End Pipeline — Qwen2.5-7B-Instruct on MATH-500
==================================================================
FIXED VERSION v2 — All mismatches with paper's GitHub resolved.

FIXES APPLIED (vs original pipeline):
  1.  Generation config: explicit temperature=1.0, top_p=1.0, top_k=0
      to override Qwen's default generation_config.json and achieve true
      greedy decoding (paper uses temperature=0.0 with vLLM).
  2.  Prompt format: exact match with paper's evaluation.py for Qwen:
        system = "You are a helpful assistant."
        user   = "Please reason step by step, and put your final answer
                  within \\boxed{}.\\n{question}"
      For TokenSkip (γ<1.0): append "<|eot_id|>{γ}<|eot_id|>" after question.
      For TokenSkip (γ=1.0): NO γ tag (same as baseline).
  3.  SFT prompt masking: labels = -100 for all prompt tokens.
      Loss is computed ONLY on the response (compressed CoT + answer).
  4.  LoRA target modules: ALL linear layers (q/k/v/o/gate/up/down_proj).
      Paper config: lora_target=all.
  5.  Training output format: "{cot}\\n\\nThe final answer is: $\\boxed{answer}$"
      Matches get_llamafactory_input.py exactly.
  6.  cutoff_len: 2048 (paper config). Was 1024.
  7.  merge_and_unload() before LoRA inference (paper's evaluation.py).
  8.  Seed = 42 with full determinism (paper's set_random_seed).
  9.  Answer extraction: robust \\boxed{} parser with nested brace support.
  10. Training batch: per_device=1, grad_accum=8 (exact paper config).
  11. Validation split: 10% held out (paper config: val_size=0.1).
  12. MATH-500 specific: max_new_tokens scaled by γ for LoRA eval
      (paper footnote 4: "we adjust its length budget to max_len×γ").
  13. attn_implementation="sdpa" for env-independent compatibility.

IMPORTANT: You MUST re-run Stages 2-4 from scratch.
           Old CoT traces and adapters use the WRONG prompt format and
           generation config.

Stages:
  1 . Load MATH train split (7,500 problems)
  2 . Qwen inference → raw CoT traces (mathtraincot.jsonl)
  3 . LLMLingua-2 compression @ mixed ratios (single pass)
  4 . LoRA fine-tune single mixed-ratio adapter (3 epochs)
  5 . MATH-500 evaluation (all prompt + truncation + LoRA methods)
  6 . Generate all 8 figures + 2 CSVs
  7 . Zip everything into a single downloadable archive
"""

# ======================================================================
#  0 . INSTALL DEPENDENCIES
# ======================================================================
import subprocess, sys, os

def pip(*pkgs, optional=False):
    try:
        subprocess.check_call(
            [sys.executable, "-m", "pip", "install", "-q", *pkgs],
            stderr=subprocess.DEVNULL if optional else None,
        )
        print(f"  [pip] installed: {' '.join(pkgs)}")
    except Exception as exc:
        if optional:
            print(f"  [pip] OPTIONAL install failed (skipping): {pkgs} - {exc}")
        else:
            raise

print("\n[0] Installing dependencies ...")
pip("transformers==4.46.3", "accelerate==0.34.2")
pip("peft==0.13.2", "llmlingua", "sentencepiece")
pip("datasets", "protobuf")
pip("seaborn", "matplotlib", "pandas", "tqdm", "scikit-learn")
pip("kgout[gdrive]", optional=True)

# ======================================================================
#  1 . IMPORTS + SEED
# ======================================================================
print("\n[1] Importing libraries ...")
import re, time, json, shutil, zipfile, argparse, pprint, traceback, glob
import random
import numpy  as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import torch
from tqdm.auto import tqdm
from datasets import load_dataset, concatenate_datasets
from transformers import (
    AutoModelForCausalLM, AutoTokenizer,
    TrainingArguments, Trainer,
)
from torch.utils.data import Dataset, Subset
from peft import LoraConfig, get_peft_model, TaskType, PeftModel
from sklearn.model_selection import train_test_split

try:
    from kgout import KgOut
    _KGOUT_AVAILABLE = True
    print("  [kgout] available")
except ImportError:
    _KGOUT_AVAILABLE = False
    print("  [kgout] not available - outputs saved to Kaggle Output tab")

try:
    from llmlingua import PromptCompressor
    _LLMLINGUA_AVAILABLE = True
    print("  [llmlingua] available")
except ImportError:
    _LLMLINGUA_AVAILABLE = False
    print("  [llmlingua] NOT available - Stage 3 will be skipped")


# -- Paper-faithful seed (evaluation.py: set_random_seed) ---------------
def set_random_seed(seed=42):
    random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    print(f"  [seed] set to {seed} with full determinism")

set_random_seed(42)

tokenizer  = None
base_model = None



[0] Installing dependencies ...
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.1/44.1 kB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.0/10.0 MB 98.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 324.4/324.4 kB 27.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 50.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0/3.0 MB 62.2 MB/s eta 0:00:00
  [pip] installed: transformers==4.46.3 accelerate==0.34.2
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 320.7/320.7 kB 13.4 MB/s eta 0:00:00
  [pip] installed: peft==0.13.2 llmlingua sentencepiece
  [pip] installed: datasets protobuf
  [pip] installed: seaborn matplotlib pandas tqdm scikit-learn
  [pip] installed: kgout[gdrive]

[1] Importing libraries ...


2026-04-12 21:35:24.480770: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1776029724.607708     106 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1776029724.646870     106 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1776029724.969994     106 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1776029724.970020     106 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1776029724.970023     106 computation_placer.cc:177] computation placer alr

  [kgout] available
  [llmlingua] available
  [seed] set to 42 with full determinism


In [2]:


# ======================================================================
#  2 . ARGPARSE
# ======================================================================
def parse_args():
    parser = argparse.ArgumentParser(
        prog="math500_tokenskip_pipeline_v2",
        description="TokenSkip Pipeline v2 (paper-faithful) — Qwen2.5 on MATH-500",
    )

    parser.add_argument("--no-kgout", action="store_true")
    parser.add_argument("--output-dir", type=str, default="/kaggle/working", metavar="DIR")
    parser.add_argument("--math500-path", type=str,
        default="/kaggle/input/math-500/math500.jsonl", metavar="PATH",
        help="Path to MATH-500 canonical .jsonl.")
    parser.add_argument("--adapter-dir", type=str, default=None, metavar="DIR",
        help="Pre-trained adapter dir. Expected sub-dir: <adapter-dir>/mathloramixed")
    parser.add_argument("--model", type=str, default="Qwen/Qwen2.5-7B-Instruct")
    parser.add_argument("--stages", type=int, nargs="+",
        default=[1, 2, 3, 4, 5, 6, 7], choices=[1, 2, 3, 4, 5, 6, 7], metavar="N")
    parser.add_argument("--skip-stages", type=int, nargs="+", default=[], metavar="N")
    parser.add_argument("--eval-only", action="store_true")
    parser.add_argument("--plots-only", action="store_true")
    parser.add_argument("--ratios", type=float, nargs="+",
        default=[0.5, 0.6, 0.7, 0.8, 0.9], metavar="R")
    parser.add_argument("--eval-batch", type=int, default=64, metavar="N")
    parser.add_argument("--max-new-tokens", type=int, default=1024, metavar="N",
        help="MATH-500 default: 1024 (paper B.1)")
    # Paper config: per_device_train_batch_size=1, gradient_accumulation_steps=8
    parser.add_argument("--train-batch", type=int, default=1,  metavar="N")
    parser.add_argument("--grad-accum",  type=int, default=8,  metavar="N")
    parser.add_argument("--epochs",      type=int, default=3,  metavar="N")
    parser.add_argument("--lr",          type=float, default=5e-5, metavar="LR")
    parser.add_argument("--lora-r",      type=int, default=8,  metavar="R")
    parser.add_argument("--lora-alpha",  type=int, default=16, metavar="A")
    parser.add_argument("--resume", action="store_true")
    parser.add_argument("--no-plots", action="store_true")
    parser.add_argument("--no-zip",   action="store_true")
    parser.add_argument("--dry-run",  action="store_true")

    args, _ = parser.parse_known_args()

    if args.eval_only:  args.stages = [5, 6, 7]
    if args.plots_only: args.stages = [6]
    if args.no_plots and 6 in args.stages: args.stages.remove(6)
    if args.no_zip   and 7 in args.stages: args.stages.remove(7)
    args.stages = sorted(set(args.stages) - set(args.skip_stages))
    return args


# ======================================================================
#  3 . RESOLVE ARGS + GLOBALS
# ======================================================================
args = parse_args()

# ── Common overrides — uncomment as needed ────────────────────
args.resume         = True
# args.stages       = [5, 6, 7]            # skip training, eval only
# args.stages       = [6, 7]               # plots + zip only
# args.no_kgout     = True                 # disable Google Drive sync
# args.dry_run      = True                 # print config, don't run
# args.adapter_dir  = "/kaggle/input/..."  # use pre-trained adapter
# ──────────────────────────────────────────────────────────────

OUTPUT_DIR     = args.output_dir
MODEL_NAME     = args.model
MAX_NEW_TOKENS = args.max_new_tokens       # 1024 for MATH-500
EVAL_BATCH     = args.eval_batch
TRAIN_BATCH    = args.train_batch
GRAD_ACCUM     = args.grad_accum
TRAIN_EPOCHS   = args.epochs
TARGET_RATIOS  = args.ratios
DEVICE         = "cuda" if torch.cuda.is_available() else "cpu"

# Baselines — updated dynamically from actual Original run in Stage 5a
ORIG_ACC    = 0.0
ORIG_TOKENS = 1.0

SUBJECTS = [
    "algebra", "counting_and_probability", "geometry",
    "intermediate_algebra", "number_theory", "prealgebra", "precalculus",
]

# ── Paper prompt (evaluation.py for Qwen) ─────────────────────
BASE_INSTRUCTION = "Please reason step by step, and put your final answer within \\boxed{}."
SYSTEM_MESSAGE   = "You are a helpful assistant."

# Prompt-based baseline variants (paper Appendix B.3 / Table 3)
PROMPT_VARIANTS = {
    "Original":    "",
    "BeConcise":   "\nBe concise.",
    "OnlyNumbers": "\nOnly use numbers or equations.",
    "AbbreWords":  "\nAbbreviate words as much as possible.",
    "LC-Prompt":   "\nPlease reduce 50% of the words in your Chain-of-Thought process.",
}

COLORS = dict(
    trunc="tomato", prompt="mediumpurple",
    lora_orig="#90CAF9", lora_guided="darkorange", lora_soft="steelblue",
    orig="gray",
)

os.makedirs(OUTPUT_DIR, exist_ok=True)

if args.dry_run:
    print("\n[dry-run] Resolved configuration:")
    pprint.pprint(vars(args))
    print(f"\n  Stages : {args.stages}")
    print(f"  Device : {DEVICE}")
    print(f"  GPU    : {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'N/A'}")
    print(f"  kgout  : {_KGOUT_AVAILABLE}")
    sys.exit(0)

print(f"\n  Device  : {DEVICE}")
print(f"  GPU     : {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'N/A'}")
print(f"  Stages  : {args.stages}")
print(f"  Ratios  : {TARGET_RATIOS}")
print(f"  Model   : {MODEL_NAME}")
print(f"  OutDir  : {OUTPUT_DIR}")


# ======================================================================
#  4 . SHARED UTILITIES
# ======================================================================
def extract_boxed(text):
    """Robust \\boxed{} extractor with nested brace support."""
    text = str(text)
    idx = text.rfind(r"\boxed{")
    if idx != -1:
        depth, start, end = 1, idx + 7, idx + 7
        while end < len(text) and depth:
            if   text[end] == "{": depth += 1
            elif text[end] == "}": depth -= 1
            end += 1
        if depth == 0:
            return text[start:end - 1]
    m = re.search(r"final answer is[:\s]*(.*)", text, re.IGNORECASE)
    return m.group(1).strip() if m else text.strip()


def normalize(ans):
    ans = str(ans).strip()
    ans = re.sub(r"\s+",       " ",  ans)
    ans = re.sub(r",(\d{3})",  r"\1", ans)
    return re.sub(r"[,\-]",    "",   ans).lower()


def is_correct(pred, gt):
    p, g = normalize(extract_boxed(pred)), normalize(extract_boxed(gt))
    if p == g:
        return True
    try:
        return abs(float(p.replace(",", "")) - float(g.replace(",", ""))) < 1e-6
    except Exception:
        return False


def make_prompt(question, method="Original"):
    """Build a chat-formatted prompt for baseline evaluation.
    Matches paper's evaluation.py Qwen branch exactly."""
    variant = PROMPT_VARIANTS.get(method, "")
    user_content = f"{BASE_INSTRUCTION}{variant}\n{question}"
    messages = [
        {"role": "system", "content": SYSTEM_MESSAGE},
        {"role": "user",   "content": user_content},
    ]
    return tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )


def make_prompt_tokenskip(question, gamma):
    """Build a γ-conditioned prompt for TokenSkip inference.
    Matches paper's evaluation.py Qwen branch exactly."""
    if gamma >= 1.0:
        user_content = f"{BASE_INSTRUCTION}\n{question}"
    else:
        user_content = f"{BASE_INSTRUCTION}\n{question}<|eot_id|>{gamma}<|eot_id|>"
    messages = [
        {"role": "system", "content": SYSTEM_MESSAGE},
        {"role": "user",   "content": user_content},
    ]
    return tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )


def save_checkpoint(results):
    path = os.path.join(OUTPUT_DIR, "mathcheckpoint.json")
    with open(path, "w") as f:
        json.dump({"results": results}, f, indent=2)
    print(f"    -> checkpoint saved ({len(results)} methods)")


def header(title):
    bar = "=" * 65
    print(f"\n{bar}\n  {title}\n{bar}")


def log(msg):
    ts = time.strftime("%H:%M:%S")
    print(f"  [{ts}] {msg}")


# ======================================================================
#  5 . BATCHED EVALUATION HELPER
# ======================================================================
def evaluate_batched(df, method="Original", max_new_tokens=None,
                     original_avg_tokens=None, model=None,
                     custom_prompts=None):
    """Batched inference + accuracy computation.
    FIX: explicit temperature=1.0, top_p=1.0, top_k=0 for true greedy."""
    global base_model
    mdl            = model if model is not None else base_model
    max_new_tokens = max_new_tokens or MAX_NEW_TOKENS
    start          = time.time()

    log(f"evaluate_batched: method={method}  n={len(df)}  "
        f"batch={EVAL_BATCH}  max_new={max_new_tokens}")

    if custom_prompts is not None:
        if len(custom_prompts) != len(df):
            raise ValueError(
                f"custom_prompts length {len(custom_prompts)} != df length {len(df)}"
            )
        prompts_indexed = list(enumerate(custom_prompts))
    else:
        prompts_indexed = [
            (seq_i, make_prompt(row["Question"], method))
            for seq_i, (_, row) in enumerate(df.iterrows())
        ]

    prompts_indexed.sort(key=lambda x: len(x[1]))
    sorted_orig_indices = [oi for oi, _ in prompts_indexed]
    sorted_prompts      = [p  for _, p  in prompts_indexed]

    all_responses    = []
    all_token_counts = []
    total_batches    = (len(sorted_prompts) + EVAL_BATCH - 1) // EVAL_BATCH

    for batch_num, bs in enumerate(
        tqdm(range(0, len(sorted_prompts), EVAL_BATCH),
             desc=f"{method}", total=total_batches, unit="batch")
    ):
        batch  = sorted_prompts[bs: bs + EVAL_BATCH]
        inputs = tokenizer(
            batch, return_tensors="pt", padding=True,
            truncation=True, max_length=2048,
        ).to(DEVICE)
        input_len = inputs["input_ids"].shape[1]

        log(f"  batch {batch_num+1}/{total_batches}  "
            f"size={len(batch)}  input_len={input_len}")

        with torch.no_grad():
            outputs = mdl.generate(
                **inputs,
                max_new_tokens=max_new_tokens,
                # ── FIX #1: True greedy decoding ──────────────
                do_sample=False,
                temperature=1.0,
                top_p=1.0,
                top_k=0,
                # ──────────────────────────────────────────────
                pad_token_id=tokenizer.pad_token_id,
                eos_token_id=tokenizer.eos_token_id,
            )

        generated    = outputs[:, input_len:]
        token_counts = (generated != tokenizer.pad_token_id).sum(dim=1).tolist()
        responses    = tokenizer.batch_decode(generated, skip_special_tokens=True)

        all_token_counts.extend(token_counts)
        all_responses.extend(responses)

        del outputs, inputs, generated
        torch.cuda.empty_cache()

    n = len(df)
    reordered_resp   = [None] * n
    reordered_tokens = [None] * n
    for sp, oi in enumerate(sorted_orig_indices):
        reordered_resp[oi]   = all_responses[sp]
        reordered_tokens[oi] = all_token_counts[sp]

    if any(r is None for r in reordered_resp):
        log("WARNING: Some reordered responses are None!")

    elapsed  = time.time() - start
    answers  = df["Answer"].tolist()
    correct  = sum(is_correct(r, g) for r, g in zip(reordered_resp, answers))
    avg_tok  = sum(reordered_tokens) / n

    metrics = {
        "Method":     method,
        "Accuracy":   round(100 * correct / n, 2),
        "Avg Tokens": round(avg_tok, 2),
        "Latency(s)": round(elapsed / n, 3),
        "Act Ratio":  round(avg_tok / original_avg_tokens, 3)
                      if original_avg_tokens else 1.0,
        "Correct":    correct,
        "Total":      n,
    }

    log(f"evaluate_batched DONE -> Acc={metrics['Accuracy']}%  "
        f"AvgTok={metrics['Avg Tokens']}  elapsed={elapsed:.1f}s")

    return metrics, reordered_resp, reordered_tokens


# ======================================================================
#  6 . SFT DATASET + COLLATOR (paper-faithful)
# ======================================================================
class CoTDataset(Dataset):
    """Paper-faithful SFT dataset for MATH.
    FIX #2: Prompt format matches paper
    FIX #3: Prompt masking — labels=-100 for prompt tokens
    FIX #5: Output format = "{cot}\\n\\nThe final answer is: $\\boxed{answer}$"
    FIX #6: cutoff_len = 2048
    """
    def __init__(self, records, tkz, max_length=2048):
        self.samples = []
        log(f"  Tokenising {len(records)} training samples (max_length={max_length}) ...")

        n_truncated = 0
        for rec in tqdm(records, desc="Tokenising", leave=False):
            gamma    = rec["gamma"]
            question = rec["problem"]

            # Extract answer from ground truth solution (robust \boxed{} extraction)
            gt_answer = extract_boxed(rec["answer"])

            # ── Build prompt (matches paper's evaluation.py Qwen branch) ──
            if gamma >= 1.0:
                user_content = f"{BASE_INSTRUCTION}\n{question}"
            else:
                user_content = f"{BASE_INSTRUCTION}\n{question}<|eot_id|>{gamma}<|eot_id|>"

            messages = [
                {"role": "system", "content": SYSTEM_MESSAGE},
                {"role": "user",   "content": user_content},
            ]
            prompt = tkz.apply_chat_template(
                messages, tokenize=False, add_generation_prompt=True
            )

            # ── Build response (matches get_llamafactory_input.py) ────────
            cot = rec["compressedcot"]
            response = f"{cot}\n\nThe final answer is: $\\boxed{{{gt_answer}}}$"

            # ── Full training sequence ────────────────────────────────────
            full_text = prompt + response + tkz.eos_token

            # ── Tokenize separately for prompt masking ────────────────────
            prompt_ids = tkz.encode(prompt, add_special_tokens=False)
            full_ids   = tkz.encode(full_text, add_special_tokens=False)

            if len(full_ids) > max_length:
                full_ids = full_ids[:max_length]
                n_truncated += 1

            prompt_len     = min(len(prompt_ids), len(full_ids))
            attention_mask = [1] * len(full_ids)

            # ── Prompt masking: labels = -100 for prompt, real ids for response ──
            labels = list(full_ids)
            for i in range(prompt_len):
                labels[i] = -100

            self.samples.append({
                "input_ids":      full_ids,
                "attention_mask": attention_mask,
                "labels":         labels,
            })

        if n_truncated > 0:
            log(f"  WARNING: {n_truncated}/{len(records)} samples truncated at {max_length} tokens")
        log(f"  Dataset ready: {len(self.samples)} samples")

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, i):
        return self.samples[i]


class SFTDataCollator:
    """Pads batches dynamically and respects pre-computed labels with -100 masking."""
    def __init__(self, pad_token_id):
        self.pad_token_id = pad_token_id

    def __call__(self, features):
        max_len = max(len(f["input_ids"]) for f in features)
        batch = {"input_ids": [], "attention_mask": [], "labels": []}
        for f in features:
            pad_len = max_len - len(f["input_ids"])
            batch["input_ids"].append(f["input_ids"]      + [self.pad_token_id] * pad_len)
            batch["attention_mask"].append(f["attention_mask"] + [0]              * pad_len)
            batch["labels"].append(f["labels"]             + [-100]              * pad_len)
        return {k: torch.tensor(v) for k, v in batch.items()}


# ======================================================================
#  7 . PLOTTER — 8 figures (includes subject heatmap for MATH)
# ======================================================================
class Plotter:
    def __init__(self, df, out=None):
        self.df  = df.copy()
        self.out = out or OUTPUT_DIR

    def _save(self, name):
        p = os.path.join(self.out, name)
        plt.tight_layout()
        plt.savefig(p, dpi=300, bbox_inches="tight")
        plt.close()
        sz = os.path.getsize(p) / 1e3
        log(f"[fig] saved -> {p} ({sz:.0f} KB)")

    def truncation_analysis(self):
        df    = self.df
        trunc = df[df.Method.str.startswith("Truncation")].sort_values("Token Savings")
        tw    = pd.concat([trunc, df[df.Method == "Original"]]).sort_values("Avg Tokens")
        fig, axes = plt.subplots(1, 3, figsize=(18, 5))

        axes[0].plot(tw["Avg Tokens"], tw["Accuracy"], "o-", color="#1565C0", lw=2, ms=8)
        for _, row in tw.iterrows():
            lbl = str(row.get("Ratio", "")) if pd.notna(row.get("Ratio", float("nan"))) else "Orig"
            axes[0].annotate(lbl, (row["Avg Tokens"], row["Accuracy"]),
                             textcoords="offset points", xytext=(5, 5), fontsize=8)
        axes[0].set_xlabel("Avg Tokens"); axes[0].set_ylabel("Accuracy %")
        axes[0].set_title("Accuracy vs Token Budget", fontsize=13, fontweight="bold")
        axes[0].grid(alpha=0.3)

        ax1 = axes[1]; ax2 = ax1.twinx()
        ax1.plot(trunc["Ratio"], trunc["Avg Tokens"], "o-", color="tab:blue", lw=2)
        ax2.plot(trunc["Ratio"], trunc["Latency(s)"], "s-", color="tab:red",  lw=2)
        ax1.set_xlabel("Truncation Ratio")
        ax1.set_ylabel("Avg Tokens",       color="tab:blue")
        ax2.set_ylabel("Latency s/sample", color="tab:red")
        ax1.tick_params(axis="y", labelcolor="tab:blue")
        ax2.tick_params(axis="y", labelcolor="tab:red")
        ax1.set_title("Tokens & Latency vs Ratio", fontsize=13, fontweight="bold")

        final = df[~df.Method.str.startswith("LoRA")]
        axes[2].scatter(final["Token Savings"], final["Accuracy"],
                        s=120, c="#1565C0", zorder=5, edgecolors="black", lw=0.5)
        for _, row in final.iterrows():
            axes[2].annotate(row["Method"], (row["Token Savings"], row["Accuracy"]),
                             textcoords="offset points", xytext=(5, 3), fontsize=7)
        axes[2].set_xlabel("Token Savings %"); axes[2].set_ylabel("Accuracy %")
        axes[2].set_title("Pareto Frontier: Accuracy vs Savings",
                          fontsize=13, fontweight="bold")
        axes[2].axvline(0, color="gray", linestyle="--", lw=0.8)
        axes[2].grid(alpha=0.3)
        self._save("math_fig1_truncation_analysis.png")

    def method_heatmap(self):
        cols  = ["Accuracy", "Avg Tokens", "Token Savings", "Latency(s)", "Efficiency Score"]
        cols  = [c for c in cols if c in self.df.columns]
        pivot = self.df.set_index("Method")[cols]
        norm  = (pivot - pivot.min()) / (pivot.max() - pivot.min() + 1e-9)
        fig, ax = plt.subplots(figsize=(10, max(6, len(self.df) * 0.38)))
        sns.heatmap(norm, annot=pivot.round(2), fmt="g", cmap="YlOrRd",
                    linewidths=0.5, ax=ax, cbar_kws={"label": "Normalized Score"})
        ax.set_title("TokenSkip Methods — MATH-500 Metric Heatmap\n(annotations = actual values)",
                     fontsize=13, fontweight="bold")
        self._save("math_fig2_method_heatmap.png")

    def token_distribution(self, all_token_counts):
        if not all_token_counts:
            log("[skip] token_distribution - no token-count data"); return
        rows    = [{"Method": m, "Tokens": c}
                   for m, counts in all_token_counts.items() for c in counts]
        dist_df = pd.DataFrame(rows)
        fig, ax = plt.subplots(figsize=(14, 5))
        sns.boxplot(data=dist_df, x="Method", y="Tokens", palette="Blues", ax=ax)
        ax.set_title("Token Length Distribution per Method — MATH-500",
                     fontsize=13, fontweight="bold")
        ax.set_xlabel(""); ax.set_ylabel("Generated Tokens")
        ax.tick_params(axis="x", rotation=25)
        self._save("math_fig3_token_distribution.png")

    def accuracy_drop_vs_savings(self):
        df     = self.df
        trunc  = df[df.Method.str.startswith("Truncation")].sort_values("Token Savings")
        soft   = df[df.Method.str.startswith("LoRASoft")].sort_values("Token Savings")
        guided = df[df.Method.str.startswith("LoRAGuided")].sort_values("Token Savings")
        fig, ax = plt.subplots(figsize=(9, 5))
        if len(trunc):
            ax.plot(trunc["Token Savings"], trunc["Accuracy Drop"], "o--",
                    color=COLORS["trunc"], lw=2, ms=7, label="Truncation")
        if len(soft):
            ax.plot(soft["Token Savings"], soft["Accuracy Drop"], "s-",
                    color=COLORS["lora_soft"], lw=2, ms=8, label="LoRA Soft")
        if len(guided):
            ax.plot(guided["Token Savings"], guided["Accuracy Drop"], "^-",
                    color=COLORS["lora_guided"], lw=2, ms=7, label="LoRA Guided")
        ax.axhline(0, linestyle=":", color="gray", lw=1.5, label="No-drop baseline")
        max_sav = df["Token Savings"].max() if len(df) else 1
        ax.fill_between([0, max_sav], 0, 3, alpha=0.06, color="green", label="Accuracy gain zone")
        ax.set_xlabel("Token Savings %", fontsize=12)
        ax.set_ylabel("Accuracy Change (pp)", fontsize=12)
        ax.set_title("Accuracy Cost of Compression — MATH-500", fontsize=13)
        ax.legend(fontsize=10); ax.grid(alpha=0.3)
        self._save("math_fig4_accuracy_drop_vs_savings.png")

    def grouped_by_ratio(self):
        df     = self.df
        ratios = TARGET_RATIOS

        def _val(col, mname):
            r = df[df.Method == mname]
            return float(r[col].values[0]) if len(r) else 0.0

        t_acc = [_val("Accuracy",      f"Truncation{r}") for r in ratios]
        s_acc = [_val("Accuracy",      f"LoRASoft{r}")   for r in ratios]
        t_sav = [_val("Token Savings", f"Truncation{r}") for r in ratios]
        s_sav = [_val("Token Savings", f"LoRASoft{r}")   for r in ratios]

        x = np.arange(len(ratios)); w = 0.35
        fig, axes = plt.subplots(1, 2, figsize=(13, 5))
        for ax, ya, yb, ylabel, title in [
            (axes[0], t_acc, s_acc, "Accuracy %",      "Accuracy by Compression Ratio"),
            (axes[1], t_sav, s_sav, "Token Savings %", "Token Savings by Ratio"),
        ]:
            ax.bar(x - w/2, ya, w, label="Truncation",
                   color=COLORS["trunc"], edgecolor="white")
            ax.bar(x + w/2, yb, w, label="LoRA Soft",
                   color=COLORS["lora_soft"], edgecolor="white")
            ax.axhline(ORIG_ACC if "Accuracy" in ylabel else 0,
                       linestyle="--", color="gray", lw=1.2)
            ax.set_xticks(x); ax.set_xticklabels([f"r={r}" for r in ratios])
            ax.set_ylabel(ylabel); ax.set_title(title)
            ax.legend(); ax.grid(axis="y", alpha=0.3)
            for i, (a, b) in enumerate(zip(ya, yb)):
                ax.text(i - w/2, a + 0.3, f"{a:.1f}",
                        ha="center", fontsize=9, fontweight="bold")
                ax.text(i + w/2, b + 0.3, f"{b:.1f}",
                        ha="center", fontsize=9, fontweight="bold")
        fig.suptitle("Truncation vs TokenSkip LoRA Soft — MATH-500", fontsize=13, y=1.01)
        self._save("math_fig5_grouped_by_ratio.png")

    def lora_triplet(self):
        df = self.df

        def _acc(m):
            r = df[df.Method == m]
            return float(r["Accuracy"].values[0]) if len(r) else 0.0

        ratios = TARGET_RATIOS
        orig   = [_acc(f"LoRA{r}")       for r in ratios]
        guided = [_acc(f"LoRAGuided{r}") for r in ratios]
        soft   = [_acc(f"LoRASoft{r}")   for r in ratios]
        x = np.arange(len(ratios)); w = 0.25
        fig, ax = plt.subplots(figsize=(10, 5))
        ax.bar(x - w, orig,   w, label="LoRA Original",
               color=COLORS["lora_orig"], edgecolor="white")
        ax.bar(x,     guided, w, label="LoRA Guided",
               color=COLORS["lora_guided"], edgecolor="white")
        ax.bar(x + w, soft,   w, label="LoRA Soft",
               color=COLORS["lora_soft"], edgecolor="white")
        ax.axhline(ORIG_ACC, linestyle="--", color="black",
                   lw=1.3, label=f"Baseline {ORIG_ACC}%")
        ax.set_xticks(x); ax.set_xticklabels([f"ratio={r}" for r in ratios])
        ax.set_ylabel("Accuracy %")
        all_vals = orig + guided + soft
        if any(v > 0 for v in all_vals):
            ax.set_ylim(max(0, min(all_vals) - 5), min(100, max(all_vals) + 5))
        ax.set_title("LoRA Variants — Accuracy by Ratio (MATH-500)",
                     fontsize=13, fontweight="bold")
        ax.legend(); ax.grid(axis="y", alpha=0.3)
        for i, (a, b, c) in enumerate(zip(orig, guided, soft)):
            ax.text(i - w, a + 0.3, f"{a:.1f}", ha="center", fontsize=9)
            ax.text(i,     b + 0.3, f"{b:.1f}", ha="center", fontsize=9)
            ax.text(i + w, c + 0.3, f"{c:.1f}", ha="center", fontsize=9)
        self._save("math_fig6_lora_triplet.png")

    def all_methods_bar(self):
        dfp = self.df.sort_values("Accuracy", ascending=True)
        colors = []
        for m in dfp.Method:
            if   m.startswith("LoRASoft"):   colors.append(COLORS["lora_soft"])
            elif m.startswith("LoRAGuided"): colors.append(COLORS["lora_guided"])
            elif m.startswith("LoRA"):       colors.append(COLORS["lora_orig"])
            elif m.startswith("Truncation"): colors.append(COLORS["trunc"])
            elif m == "Original":            colors.append(COLORS["orig"])
            else:                            colors.append(COLORS["prompt"])
        fig, ax = plt.subplots(figsize=(9, max(6, len(dfp) * 0.4)))
        bars = ax.barh(dfp.Method, dfp.Accuracy, color=colors, edgecolor="white")
        ax.axvline(ORIG_ACC, linestyle="--", color="black", lw=1.2)
        ax.set_xlabel("Accuracy %")
        ax.set_xlim(0, dfp.Accuracy.max() + 8)
        ax.set_title("All Methods Ranked by Accuracy — MATH-500",
                     fontsize=13, fontweight="bold")
        for bar, val in zip(bars, dfp.Accuracy):
            ax.text(bar.get_width() + 0.3,
                    bar.get_y() + bar.get_height() / 2,
                    f"{val:.1f}%", va="center", fontsize=9)
        patches = [
            mpatches.Patch(color=COLORS["orig"],        label="Original"),
            mpatches.Patch(color=COLORS["prompt"],      label="Prompt Variant"),
            mpatches.Patch(color=COLORS["trunc"],       label="Truncation"),
            mpatches.Patch(color=COLORS["lora_orig"],   label="LoRA"),
            mpatches.Patch(color=COLORS["lora_guided"], label="LoRA Guided"),
            mpatches.Patch(color=COLORS["lora_soft"],   label="LoRA Soft"),
        ]
        ax.legend(handles=patches, loc="lower right", fontsize=9)
        self._save("math_fig7_all_methods_bar.png")

    def subject_accuracy_heatmap(self, results_by_subject):
        if not results_by_subject:
            log("[skip] subject_accuracy_heatmap - no subject data"); return
        subj_df = pd.DataFrame(results_by_subject).T
        fig, ax = plt.subplots(
            figsize=(max(10, len(subj_df.columns)), max(5, len(subj_df) * 0.4))
        )
        sns.heatmap(subj_df, annot=True, fmt=".1f", cmap="RdYlGn",
                    linewidths=0.4, ax=ax, cbar_kws={"label": "Accuracy %"})
        ax.set_title("Per-Subject Accuracy Heatmap — MATH-500",
                     fontsize=13, fontweight="bold")
        ax.set_xlabel("Subject"); ax.set_ylabel("Method")
        self._save("math_fig8_subject_accuracy.png")

    def run_all(self, all_token_counts=None, results_by_subject=None):
        header("STAGE 6 . Generating all 8 figures")
        self.truncation_analysis()
        self.method_heatmap()
        self.token_distribution(all_token_counts or {})
        self.accuracy_drop_vs_savings()
        self.grouped_by_ratio()
        self.lora_triplet()
        self.all_methods_bar()
        self.subject_accuracy_heatmap(results_by_subject or {})
        log("All 8 figures complete.")


# ======================================================================
#  8 . MAIN PIPELINE
# ======================================================================
def run_pipeline():
    global tokenizer, base_model, ORIG_ACC, ORIG_TOKENS

    # -- kgout setup -------------------------------------------------------
    use_kgout = False
    kg        = None

    if not args.no_kgout and _KGOUT_AVAILABLE:
        try:
            all_json = glob.glob("/kaggle/input/**/*.json", recursive=True)
            oauth_files = [f for f in all_json if "kgout_token" in f]
            sa_files    = [f for f in all_json if "youtube-tracker" in f]
            cred_path   = oauth_files[0] if oauth_files else (
                          sa_files[0]    if sa_files    else None)
            if not cred_path:
                raise FileNotFoundError("No credential JSON found in /kaggle/input/.")
            log(f"kgout: credential -> {cred_path}")
            kg = KgOut(
                folder_id="11IYp2WZOh5fuIXpY432Kz2usG4XorBAu",
                credentials=cred_path,
                interval=180,
            ).start()
            use_kgout = True
            log("kgout started - syncing to Google Drive.")
        except Exception as exc:
            log(f"WARNING: kgout failed ({exc}). Continuing without it.")
            use_kgout = False
    elif args.no_kgout:
        log("kgout disabled by --no-kgout flag.")
    else:
        log("kgout disabled (package not installed).")

    # -- Load model + tokenizer --------------------------------------------
    if any(s in args.stages for s in [2, 4, 5]):
        header("LOADING MODEL & TOKENIZER")
        log(f"Loading tokenizer from {MODEL_NAME} ...")
        tokenizer = AutoTokenizer.from_pretrained(
            MODEL_NAME, trust_remote_code=True, padding_side="left"
        )
        if tokenizer.pad_token is None:
            tokenizer.pad_token    = tokenizer.eos_token
            tokenizer.pad_token_id = tokenizer.eos_token_id
        log(f"Tokenizer ready (vocab={tokenizer.vocab_size})")

        log(f"Loading base model from {MODEL_NAME} ...")
        base_model = AutoModelForCausalLM.from_pretrained(
            MODEL_NAME,
            torch_dtype=torch.bfloat16,
            device_map="auto",
            trust_remote_code=True,
            attn_implementation="sdpa",
        )
        base_model.eval()
        log(f"Model on: {next(base_model.parameters()).device}")
        if torch.cuda.is_available():
            mem = torch.cuda.memory_allocated() / 1e9
            log(f"GPU memory used after model load: {mem:.2f} GB")

    # ==================================================================
    #  STAGE 1 . LOAD MATH TRAIN SPLIT
    # ==================================================================
    train_df = None
    if 1 in args.stages:
        header("STAGE 1 . Loading MATH train split")
        all_splits = []
        for subj in SUBJECTS:
            log(f"Loading subject: {subj} ...")
            ds = load_dataset("EleutherAI/hendrycks_math", subj, split="train")
            log(f"  {subj:<30s}  {len(ds)} problems")
            all_splits.append(ds)
        math_ds  = concatenate_datasets(all_splits)
        train_df = pd.DataFrame({
            "Question": [ex["problem"]  for ex in math_ds],
            "Answer":   [ex["solution"] for ex in math_ds],
            "Level":    [ex["level"]    for ex in math_ds],
            "Type":     [ex["type"]     for ex in math_ds],
        }).reset_index(drop=True)
        log(f"Train set shape: {train_df.shape}")
        log(f"Level distribution:\n{train_df.Level.value_counts().to_string()}")

    # ==================================================================
    #  STAGE 2 . GENERATE RAW CoT TRACES
    # ==================================================================
    if 2 in args.stages:
        header("STAGE 2 . Generating raw CoT traces on MATH train split")
        if train_df is None:
            raise RuntimeError("Stage 2 requires Stage 1.")

        COT_PATH = os.path.join(OUTPUT_DIR, "mathtraincot.jsonl")
        done_ids = set()

        if os.path.exists(COT_PATH) and args.resume:
            with open(COT_PATH) as f:
                done_ids = {json.loads(l)["id"] for l in f}
            log(f"Resuming - {len(done_ids)}/{len(train_df)} already done")
        else:
            if os.path.exists(COT_PATH):
                log("Deleting old CoT traces (incompatible with v2 fixes).")
                os.remove(COT_PATH)
            log(f"Starting fresh - {len(train_df)} problems to process")

        remaining_mask = ~train_df.index.isin(done_ids)
        remaining_df   = train_df[remaining_mask].reset_index(drop=True)
        remaining_orig = train_df[remaining_mask].index.tolist()

        if len(remaining_df) == 0:
            log("All CoT traces already exist - skipping inference.")
        else:
            log(f"Running inference on {len(remaining_df)} problems ...")
            _, responses, token_counts = evaluate_batched(
                remaining_df, method="Original"
            )
            with open(COT_PATH, "a") as f:
                for li, (resp, tc) in enumerate(zip(responses, token_counts)):
                    orig_idx = remaining_orig[li]
                    row      = train_df.iloc[orig_idx]
                    f.write(json.dumps({
                        "id":         int(orig_idx),
                        "problem":    row["Question"],
                        "answer":     row["Answer"],
                        "fullcot":    resp,
                        "tokencount": tc,
                        "level":      row["Level"],
                        "subject":    row["Type"],
                    }) + "\n")
            log(f"Saved -> {COT_PATH}")

        with open(COT_PATH) as f:
            cot_records = [json.loads(l) for l in f]
        avg_tok = sum(r["tokencount"] for r in cot_records) / len(cot_records)
        log(f"CoT file: {len(cot_records)} records | avg tokens: {avg_tok:.1f}")

    # ==================================================================
    #  STAGE 3 . LLMLingua-2 COMPRESSION (MIXED RATIO)
    # ==================================================================
    if 3 in args.stages:
        header("STAGE 3 . LLMLingua-2 compression (mixed ratio)")
        if not _LLMLINGUA_AVAILABLE:
            log("ERROR: llmlingua not installed - skipping Stage 3.")
        else:
            COT_PATH = os.path.join(OUTPUT_DIR, "mathtraincot.jsonl")
            if not os.path.exists(COT_PATH):
                raise FileNotFoundError(f"CoT file not found: {COT_PATH}\nRun Stage 2 first.")
            with open(COT_PATH) as f:
                cot_records = [json.loads(l) for l in f]
            log(f"Loaded {len(cot_records)} CoT records")

            # Answer filtering (paper §3.2)
            n_before    = len(cot_records)
            cot_records = [r for r in cot_records if is_correct(r["fullcot"], r["answer"])]
            log(f"Answer filtering: {n_before} -> {len(cot_records)} records "
                f"({n_before - len(cot_records)} incorrect removed)")

            out_path = os.path.join(OUTPUT_DIR, "mathtraincompressed.jsonl")

            if os.path.exists(out_path) and args.resume:
                with open(out_path) as f:
                    n_exist = sum(1 for _ in f)
                log(f"Compressed file already exists ({n_exist} records) - skipping.")
            else:
                TRAIN_RATIOS = [0.5, 0.6, 0.7, 0.8, 0.9, 1.0]

                log("Loading LLMLingua-2 on GPU ...")
                compressor = PromptCompressor(
                    model_name="microsoft/llmlingua-2-xlm-roberta-large-meetingbank",
                    use_llmlingua2=True, device_map="cuda",
                )
                log("LLMLingua-2 ready!")
                log(f"Compressing {len(cot_records)} samples with mixed ratios ...")
                t0 = time.time(); n_errors = 0; n_original = 0

                with open(out_path, "w") as fout:
                    for rec in tqdm(cot_records, desc="LLMLingua mixed"):
                        gamma = float(np.random.choice(TRAIN_RATIOS))
                        if gamma == 1.0:
                            compressed = rec["fullcot"]
                            n_original += 1
                        else:
                            try:
                                result = compressor.compress_prompt(
                                    rec["fullcot"], rate=gamma,
                                )
                                compressed = result["compressed_prompt"]
                            except Exception:
                                n_errors  += 1
                                compressed = rec["fullcot"]
                        fout.write(json.dumps({
                            "id":             rec["id"],
                            "problem":        rec["problem"],
                            "answer":         rec["answer"],
                            "compressedcot":  compressed,
                            "originaltokens": rec["tokencount"],
                            "gamma":          gamma,
                            "level":          rec.get("level", ""),
                            "subject":        rec.get("subject", ""),
                        }) + "\n")

                elapsed = (time.time() - t0) / 60
                log(f"Compression done in {elapsed:.1f} min "
                    f"(γ=1.0 samples={n_original} fallbacks={n_errors}) -> {out_path}")
                del compressor
                torch.cuda.empty_cache()

    # ==================================================================
    #  STAGE 4 . LoRA TRAINING (PAPER-FAITHFUL)
    # ==================================================================
    if 4 in args.stages:
        header("STAGE 4 . LoRA fine-tuning (paper-faithful)")

        adapter_dir = os.path.join(
            args.adapter_dir if args.adapter_dir else OUTPUT_DIR,
            "mathloramixed"
        )
        zip_path = adapter_dir + ".zip"

        if (os.path.exists(zip_path) or os.path.isdir(adapter_dir)) and args.resume:
            log("Mixed-ratio adapter already exists - skipping Stage 4.")
        else:
            compressed_path = os.path.join(OUTPUT_DIR, "mathtraincompressed.jsonl")
            if not os.path.exists(compressed_path):
                raise FileNotFoundError(
                    f"Compressed file not found: {compressed_path}\nRun Stage 3 first.")

            with open(compressed_path) as f:
                records = [json.loads(l) for l in f]
            log(f"Loaded {len(records)} mixed-ratio training records")

            print(f"\n{'--'*32}")
            print(f"  LoRA Training (paper-faithful) — MATH")
            print(f"  epochs={TRAIN_EPOCHS}  lr={args.lr}")
            print(f"  r={args.lora_r}  alpha={args.lora_alpha}")
            print(f"  batch={TRAIN_BATCH}  grad_accum={GRAD_ACCUM}")
            print(f"  target=ALL linear layers  cutoff=2048")
            print(f"  prompt masking=ON  val_split=10%")
            print(f"  output -> {adapter_dir}")
            print(f"{'--'*32}")

            # ── FIX #4: lora_target = all linear layers ───────────────
            lora_cfg = LoraConfig(
                r=args.lora_r,
                lora_alpha=args.lora_alpha,
                target_modules=[
                    "q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj",
                ],
                lora_dropout=0.05,
                bias="none",
                task_type=TaskType.CAUSAL_LM,
            )
            lora_model = get_peft_model(base_model, lora_cfg)
            lora_model.print_trainable_parameters()

            dataset = CoTDataset(records, tokenizer, max_length=2048)

            # ── FIX #11: 10% validation split ─────────────────────────
            train_idx, val_idx = train_test_split(
                range(len(dataset)), test_size=0.1, random_state=42
            )
            train_dataset = Subset(dataset, train_idx)
            val_dataset   = Subset(dataset, val_idx)
            log(f"Train: {len(train_dataset)}  Val: {len(val_dataset)}")

            collator = SFTDataCollator(pad_token_id=tokenizer.pad_token_id)

            train_args = TrainingArguments(
                output_dir=os.path.join(OUTPUT_DIR, "mathlorackptmixed"),
                num_train_epochs=TRAIN_EPOCHS,
                per_device_train_batch_size=TRAIN_BATCH,
                gradient_accumulation_steps=GRAD_ACCUM,
                learning_rate=args.lr,
                warmup_ratio=0.1,
                lr_scheduler_type="cosine",
                optim="adamw_torch",
                bf16=True,
                logging_steps=10,
                eval_strategy="steps",
                eval_steps=300,
                per_device_eval_batch_size=1,
                save_strategy="steps",
                save_steps=300,
                save_total_limit=2,
                load_best_model_at_end=True,
                metric_for_best_model="eval_loss",
                greater_is_better=False,
                report_to="none",
                dataloader_num_workers=2,
                seed=42,
            )
            trainer = Trainer(
                model=lora_model,
                args=train_args,
                train_dataset=train_dataset,
                eval_dataset=val_dataset,
                data_collator=collator,
            )
            log("Starting trainer ...")
            trainer.train()
            log("Training complete")

            os.makedirs(adapter_dir, exist_ok=True)
            lora_model.save_pretrained(adapter_dir)
            tokenizer.save_pretrained(adapter_dir)
            log(f"Adapter saved -> {adapter_dir}")

            shutil.make_archive(adapter_dir, "zip", adapter_dir)
            sz = os.path.getsize(zip_path) / 1e6
            log(f"Adapter ZIP -> {zip_path}  ({sz:.1f} MB)")

            log("Unloading LoRA adapter - restoring base_model ...")
            try:
                base_model = lora_model.unload()
                if base_model is None:
                    raise AttributeError("unload() returned None")
            except Exception:
                base_model = lora_model.base_model.model
                if hasattr(base_model, "peft_config"):
                    del base_model.peft_config

            del lora_model, trainer, dataset, train_dataset, val_dataset
            torch.cuda.empty_cache()
            base_model.eval()
            log("base_model restored and set to eval mode.")
            # Clean up training checkpoints (optimizer states eat disk)
            ckpt_dir = os.path.join(OUTPUT_DIR, "mathlorackptmixed")
            if os.path.isdir(ckpt_dir):
                shutil.rmtree(ckpt_dir)
                log(f"Deleted training checkpoints: {ckpt_dir}")
            log("MIXED-RATIO ADAPTER TRAINED AND SAVED")

    # ==================================================================
    #  STAGE 5 . MATH-500 EVALUATION
    # ==================================================================
    results_df   = None
    all_tok_dict = {}
    subj_results = {}

    if 5 in args.stages:
        header("STAGE 5 . MATH-500 Evaluation")

        # Load MATH-500 test set
        if os.path.exists(args.math500_path):
            log(f"Loading MATH-500 canonical: {args.math500_path}")
            with open(args.math500_path) as f:
                m500 = [json.loads(l) for l in f]
            test_df = pd.DataFrame({
                "Question": [x["problem"]       for x in m500],
                "Answer":   [x["solution"]      for x in m500],
                "Level":    [x.get("level", "") for x in m500],
                "Type":     [x.get("subject",  "") for x in m500],
            })
            log(f"MATH-500 canonical loaded: {len(test_df)} problems")
        else:
            log("Local file not found. Loading HuggingFaceH4/MATH-500 ...")
            ds = load_dataset("HuggingFaceH4/MATH-500", split="test")
            test_df = pd.DataFrame({
                "Question": [ex["problem"]  for ex in ds],
                "Answer":   [ex["solution"] for ex in ds],
                "Level":    [ex.get("level", "") for ex in ds],
                "Type":     [ex.get("subject",  "") for ex in ds],
            }).reset_index(drop=True)
            log(f"MATH-500 loaded: {len(test_df)} problems")

        results   = []
        done_meth = set()
        CHECKPOINT = os.path.join(OUTPUT_DIR, "mathcheckpoint.json")
        if os.path.exists(CHECKPOINT) and args.resume:
            with open(CHECKPOINT) as f:
                results = json.load(f).get("results", [])
            done_meth = {r["Method"] for r in results}
            log(f"Checkpoint loaded - {len(done_meth)} methods done: {sorted(done_meth)}")
            _orig = next((r for r in results if r["Method"] == "Original"), None)
            if _orig:
                ORIG_TOKENS = _orig["Avg Tokens"]
                ORIG_ACC    = _orig["Accuracy"]
                log(f"Baselines restored: ORIG_TOKENS={ORIG_TOKENS}  ORIG_ACC={ORIG_ACC}%")
        else:
            log("Starting evaluation from scratch.")

        def record_subj_acc(method, responses):
            cb, tb = {}, {}
            for resp, row in zip(responses, test_df.itertuples()):
                s      = row.Type
                cb[s]  = cb.get(s, 0) + int(is_correct(resp, row.Answer))
                tb[s]  = tb.get(s, 0) + 1
            subj_results[method] = {
                s: round(100 * cb[s] / tb[s], 1) for s in tb
            }

        def run_method(name, model=None, prompt_override=None):
            if name in done_meth:
                log(f"[{name}] checkpoint hit - skipping."); return
            log(f"Starting evaluation: {name} ...")
            row, resp, tok = evaluate_batched(
                test_df,
                method=prompt_override or name,
                original_avg_tokens=ORIG_TOKENS,
                model=model,
            )
            row["Method"] = name
            results.append(row)
            all_tok_dict[name] = tok
            record_subj_acc(name, resp)
            done_meth.add(name)
            save_checkpoint(results)
            log(f"[{name}]  Acc={row['Accuracy']}%  "
                f"AvgTok={row['Avg Tokens']}  Latency={row['Latency(s)']}s")

        # -- 5a . Prompt methods -------------------------------------------
        header("  5a . Prompt-engineering methods")
        run_method("Original")

        orig_row = next((r for r in results if r["Method"] == "Original"), None)
        if orig_row:
            ORIG_TOKENS = orig_row["Avg Tokens"]
            ORIG_ACC    = orig_row["Accuracy"]
            log(f"Baselines updated: ORIG_TOKENS={ORIG_TOKENS}  ORIG_ACC={ORIG_ACC}%")

        for pm in ["BeConcise", "OnlyNumbers", "AbbreWords", "LC-Prompt"]:
            run_method(pm)

        # -- 5b . Truncation -----------------------------------------------
        header("  5b . Truncation methods")
        for ratio in TARGET_RATIOS:
            mname = f"Truncation{ratio}"
            if mname in done_meth:
                log(f"[{mname}] checkpoint hit - skipping."); continue

            log(f"Evaluating truncation at ratio={ratio} "
                f"(max_new_tokens={int(MAX_NEW_TOKENS * ratio)}) ...")
            row, resp, tok = evaluate_batched(
                test_df, method="Original",
                max_new_tokens=int(MAX_NEW_TOKENS * ratio),
                original_avg_tokens=ORIG_TOKENS,
            )
            row["Method"] = mname
            row["Ratio"]  = ratio
            results.append(row)
            all_tok_dict[mname] = tok
            record_subj_acc(mname, resp)
            done_meth.add(mname)
            save_checkpoint(results)
            log(f"[{mname}]  Acc={row['Accuracy']}%  AvgTok={row['Avg Tokens']}")

        # -- 5c . LLMLingua ------------------------------------------------
        header("  5c . LLMLingua compressed evaluation")
        if _LLMLINGUA_AVAILABLE:
            log("Loading LLMLingua-2 for test compression ...")
            compressor = PromptCompressor(
                model_name="microsoft/llmlingua-2-xlm-roberta-large-meetingbank",
                use_llmlingua2=True, device_map="cuda",
            )
            for ratio in TARGET_RATIOS:
                mname = f"LLMLingua{ratio}"
                if mname in done_meth:
                    log(f"[{mname}] checkpoint hit - skipping."); continue
                log(f"Compressing MATH-500 test prompts at ratio={ratio} ...")
                compressed_prompts = []
                for _, row in test_df.iterrows():
                    original = f"{BASE_INSTRUCTION}\n{row['Question']}"
                    try:
                        result     = compressor.compress_prompt(
                            original, rate=ratio)
                        compressed = result["compressed_prompt"]
                    except Exception:
                        compressed = original
                    messages = [
                        {"role": "system", "content": SYSTEM_MESSAGE},
                        {"role": "user",   "content": compressed},
                    ]
                    compressed_prompts.append(
                        tokenizer.apply_chat_template(
                            messages, tokenize=False, add_generation_prompt=True))
                row_r, resp, tok = evaluate_batched(
                    test_df, method=mname,
                    original_avg_tokens=ORIG_TOKENS,
                    custom_prompts=compressed_prompts,
                )
                row_r["Ratio"] = ratio
                results.append(row_r)
                all_tok_dict[mname] = tok
                record_subj_acc(mname, resp)
                done_meth.add(mname)
                save_checkpoint(results)
                log(f"[{mname}]  Acc={row_r['Accuracy']}%  AvgTok={row_r['Avg Tokens']}")
            del compressor
            torch.cuda.empty_cache()
        else:
            log("LLMLingua not available - skipping 5c.")

        # -- 5d . LoRA adapter evaluation ----------------------------------
        header("  5d . LoRA adapter evaluation (mixed-ratio)")

        adapter_dir = os.path.join(
            args.adapter_dir if args.adapter_dir else OUTPUT_DIR,
            "mathloramixed"
        )
        zip_path = adapter_dir + ".zip"

        if not os.path.isdir(adapter_dir) and os.path.exists(zip_path):
            log(f"Unpacking adapter ZIP: {zip_path} ...")
            shutil.unpack_archive(zip_path, adapter_dir)

        if not os.path.isdir(adapter_dir):
            log("[SKIP] mixed-ratio adapter not found.")
            log("  (Run Stage 4, or supply --adapter-dir)")
        else:
            # ── FIX #7: merge_and_unload() before inference ───────────
            log(f"Loading mixed-ratio LoRA adapter from {adapter_dir} ...")
            lora_model = PeftModel.from_pretrained(base_model, adapter_dir)
            merged_model = lora_model.merge_and_unload()
            merged_model.eval()
            log("Adapter merged and ready for inference")

            for ratio in TARGET_RATIOS:
                mname = f"LoRA{ratio}"
                if mname in done_meth:
                    log(f"[{mname}] checkpoint hit - skipping."); continue

                # ── FIX #12: MATH-500 specific — max_new_tokens × γ ──
                # Paper footnote 4: "we adjust its length budget to
                # max_len×γ" for MATH-500 (NOT for GSM8K)
                scaled_tokens = int(MAX_NEW_TOKENS * ratio)
                log(f"Evaluating {mname} with γ={ratio} "
                    f"(max_new_tokens={scaled_tokens}) ...")

                tokenskip_prompts = [
                    make_prompt_tokenskip(row["Question"], ratio)
                    for _, row in test_df.iterrows()
                ]
                row, resp, tok = evaluate_batched(
                    test_df, method=mname,
                    original_avg_tokens=ORIG_TOKENS,
                    model=merged_model,
                    custom_prompts=tokenskip_prompts,
                    max_new_tokens=scaled_tokens,
                )
                row["Method"] = mname
                row["Ratio"]  = ratio
                results.append(row)
                all_tok_dict[mname] = tok
                record_subj_acc(mname, resp)
                done_meth.add(mname)
                save_checkpoint(results)
                log(f"[{mname}]  Acc={row['Accuracy']}%  AvgTok={row['Avg Tokens']}")

                # ── Guided/Soft variants ──────────────────────────────
                for suffix, variant_key in [("Guided", "BeConcise"),
                                            ("Soft",   "LC-Prompt")]:
                    mname2 = f"LoRA{suffix}{ratio}"
                    if mname2 in done_meth:
                        log(f"[{mname2}] checkpoint hit - skipping."); continue
                    log(f"Evaluating {mname2} with γ={ratio} + {variant_key} "
                        f"(max_new_tokens={scaled_tokens}) ...")

                    variant_text = PROMPT_VARIANTS[variant_key]
                    guided_prompts = []
                    for _, row in test_df.iterrows():
                        q = row["Question"]
                        if ratio >= 1.0:
                            user_content = f"{BASE_INSTRUCTION}{variant_text}\n{q}"
                        else:
                            user_content = f"{BASE_INSTRUCTION}{variant_text}\n{q}<|eot_id|>{ratio}<|eot_id|>"
                        messages = [
                            {"role": "system", "content": SYSTEM_MESSAGE},
                            {"role": "user",   "content": user_content},
                        ]
                        guided_prompts.append(tokenizer.apply_chat_template(
                            messages, tokenize=False, add_generation_prompt=True))

                    row2, resp2, tok2 = evaluate_batched(
                        test_df, method=mname2,
                        original_avg_tokens=ORIG_TOKENS,
                        model=merged_model,
                        custom_prompts=guided_prompts,
                        max_new_tokens=scaled_tokens,
                    )
                    row2["Method"] = mname2
                    row2["Ratio"]  = ratio
                    results.append(row2)
                    all_tok_dict[mname2] = tok2
                    record_subj_acc(mname2, resp2)
                    done_meth.add(mname2)
                    save_checkpoint(results)
                    log(f"[{mname2}]  Acc={row2['Accuracy']}%  AvgTok={row2['Avg Tokens']}")

            # -- 5e . EXTENSION: Adaptive γ based on difficulty level ─────
            # Maps MATH difficulty Level 1–5 → γ 0.5–0.9
            # Easier problems get more compression, harder problems less.
            # No retraining needed — same adapter handles all γ dynamically.
            header("  5e . Adaptive Compression Ratio (extension)")

            LEVEL_TO_GAMMA = {
                "Level 1": 0.5,  1: 0.5,  "1": 0.5,
                "Level 2": 0.6,  2: 0.6,  "2": 0.6,
                "Level 3": 0.7,  3: 0.7,  "3": 0.7,
                "Level 4": 0.8,  4: 0.8,  "4": 0.8,
                "Level 5": 0.9,  5: 0.9,  "5": 0.9,
            }

            mname_adaptive = "LoRAAdaptive"
            if mname_adaptive in done_meth:
                log(f"[{mname_adaptive}] checkpoint hit - skipping.")
            elif "Level" not in test_df.columns or test_df["Level"].isna().all():
                log(f"[{mname_adaptive}] SKIP - no Level column in test data.")
            else:
                log(f"Evaluating {mname_adaptive} with per-problem γ based on difficulty ...")
                log(f"  Level→γ mapping: {LEVEL_TO_GAMMA}")

                adaptive_prompts = []
                adaptive_max_tokens_list = []
                for _, row in test_df.iterrows():
                    level = str(row.get("Level", "Level 3"))
                    gamma = LEVEL_TO_GAMMA.get(level, 0.7)  # default to 0.7 if unknown
                    adaptive_prompts.append(
                        make_prompt_tokenskip(row["Question"], gamma)
                    )
                    # Paper footnote 4: max_len × γ for MATH-500
                    adaptive_max_tokens_list.append(int(MAX_NEW_TOKENS * gamma))

                # Since evaluate_batched takes a single max_new_tokens, we use
                # the maximum across all samples (so no sample gets truncated
                # prematurely by the batch limit). The actual compression comes
                # from the model learning to stop early based on the γ signal.
                max_adaptive_tokens = max(adaptive_max_tokens_list)
                log(f"  Using max_new_tokens={max_adaptive_tokens} "
                    f"(max across all adaptive γ values)")

                row_a, resp_a, tok_a = evaluate_batched(
                    test_df, method=mname_adaptive,
                    original_avg_tokens=ORIG_TOKENS,
                    model=merged_model,
                    custom_prompts=adaptive_prompts,
                    max_new_tokens=max_adaptive_tokens,
                )
                row_a["Method"] = mname_adaptive
                row_a["Ratio"]  = "adaptive"
                results.append(row_a)
                all_tok_dict[mname_adaptive] = tok_a
                record_subj_acc(mname_adaptive, resp_a)
                done_meth.add(mname_adaptive)
                save_checkpoint(results)
                log(f"[{mname_adaptive}]  Acc={row_a['Accuracy']}%  "
                    f"AvgTok={row_a['Avg Tokens']}")

                # Per-level breakdown for adaptive
                level_stats = {}
                answers = test_df["Answer"].tolist()
                levels  = test_df["Level"].tolist()
                for i, (resp, gt, lvl) in enumerate(zip(resp_a, answers, levels)):
                    lvl = str(lvl)
                    if lvl not in level_stats:
                        level_stats[lvl] = {"correct": 0, "total": 0,
                                            "tokens": [], "gamma": LEVEL_TO_GAMMA.get(lvl, 0.7)}
                    level_stats[lvl]["total"]   += 1
                    level_stats[lvl]["correct"] += int(is_correct(resp, gt))
                    level_stats[lvl]["tokens"].append(tok_a[i])

                log("  Adaptive per-level breakdown:")
                for lvl in sorted(level_stats.keys()):
                    s = level_stats[lvl]
                    acc = 100 * s["correct"] / s["total"] if s["total"] else 0
                    avg = sum(s["tokens"]) / len(s["tokens"]) if s["tokens"] else 0
                    log(f"    {lvl} (γ={s['gamma']}): "
                        f"Acc={acc:.1f}%  AvgTok={avg:.1f}  n={s['total']}")

            del merged_model, lora_model
            torch.cuda.empty_cache()
            log("LoRA evaluation done - GPU memory freed.")

        # -- Build results DataFrame ---------------------------------------
        results_df = pd.DataFrame(results)
        results_df["Token Savings"]    = (
            (ORIG_TOKENS - results_df["Avg Tokens"]) / ORIG_TOKENS * 100
        ).round(2)
        results_df["Accuracy Drop"]    = (
            results_df["Accuracy"] - ORIG_ACC
        ).round(2)
        results_df["Efficiency Score"] = (
            results_df["Accuracy"] / results_df["Avg Tokens"] * 100
        ).round(4)

        base_csv  = os.path.join(OUTPUT_DIR, "mathresults.csv")
        final_csv = os.path.join(OUTPUT_DIR, "mathresultsfinal.csv")
        results_df[~results_df.Method.str.startswith("LoRA")].to_csv(base_csv, index=False)
        results_df.to_csv(final_csv, index=False)
        log(f"CSVs saved:\n    {base_csv}\n    {final_csv}")

        summary_cols = ["Method", "Accuracy", "Avg Tokens", "Token Savings", "Latency(s)"]
        print("\n" + results_df[summary_cols].to_string(index=False))

    # ==================================================================
    #  STAGE 6 . GENERATE ALL 8 FIGURES
    # ==================================================================
    if 6 in args.stages:
        if results_df is None:
            final_csv = os.path.join(OUTPUT_DIR, "mathresultsfinal.csv")
            if not os.path.exists(final_csv):
                raise FileNotFoundError(
                    f"Results CSV not found: {final_csv}\nRun Stage 5 first.")
            results_df = pd.read_csv(final_csv)
            log(f"Loaded results from {final_csv}  ({len(results_df)} rows)")
            if not all_tok_dict:
                log("Note: per-method token distributions unavailable (Fig 3 skipped).")

        Plotter(results_df).run_all(
            all_token_counts=all_tok_dict if all_tok_dict else None,
            results_by_subject=subj_results if subj_results else None,
        )

    # ==================================================================
    #  STAGE 7 . ZIP EVERYTHING
    # ==================================================================
    if 7 in args.stages:
        header("STAGE 7 . Zipping all outputs")
        ZIP_FILE = os.path.join(OUTPUT_DIR, "math_full_outputs_7B.zip")
        n_files  = 0
        with zipfile.ZipFile(ZIP_FILE, "w", zipfile.ZIP_DEFLATED) as zf:
            for root, dirs, files in os.walk(OUTPUT_DIR):
                dirs[:] = [d for d in dirs if not d.startswith("mathlorackpt")]
                for fname in sorted(files):
                    if fname.endswith(".zip"):
                        continue
                    fpath   = os.path.join(root, fname)
                    arcname = os.path.relpath(fpath, OUTPUT_DIR)
                    zf.write(fpath, arcname)
                    n_files += 1
                    log(f"  added to ZIP: {arcname}")
        sz = os.path.getsize(ZIP_FILE) / 1e6
        log(f"Master ZIP -> {ZIP_FILE}  ({sz:.1f} MB, {n_files} files)")

    # -- stop kgout --------------------------------------------------------
    if use_kgout and kg is not None:
        try:
            kg.stop()
            time.sleep(290)
            log("kgout stopped.")
        except Exception as exc:
            log(f"kgout stop warning: {exc}")

    # ==================================================================
    #  FINAL MANIFEST
    # ==================================================================
    print("\n" + "="*65 + "\n  OUTPUT MANIFEST\n" + "="*65)
    total_size = 0
    for root, _, files in os.walk(OUTPUT_DIR):
        for fname in sorted(files):
            fpath  = os.path.join(root, fname)
            sz_mb  = os.path.getsize(fpath) / 1e6
            total_size += sz_mb
            relpath = os.path.relpath(fpath, OUTPUT_DIR)
            print(f"  {relpath:<55s}  {sz_mb:7.2f} MB")
    print(f"\n  {'TOTAL':<55s}  {total_size:7.2f} MB")
    print("="*65)
    print("\n  ALL STAGES COMPLETE")

    if use_kgout:
        print("  Every file above is synced to Google Drive.")
    else:
        print("  Files are available in the /kaggle/working Output tab.")
        print("  Download math_full_outputs_7B.zip for the full bundle.")
        try:
            from IPython.display import FileLinks, display
            display(FileLinks(OUTPUT_DIR))
        except Exception:
            pass


# ======================================================================
#  ENTRY POINT
# ======================================================================

# -- Copy CoT traces from dataset input if available -------------------
import glob as _glob, shutil as _shutil
_cot_dst = "/kaggle/working/mathtraincot.jsonl"
if not os.path.exists(_cot_dst):
    _matches = _glob.glob("/kaggle/input/**/mathtraincot.jsonl", recursive=True)
    if _matches:
        _shutil.copy(_matches[0], _cot_dst)
        print(f"Copied mathtraincot.jsonl from dataset "
              f"({os.path.getsize(_cot_dst)/1e6:.1f} MB)")
    else:
        print("mathtraincot.jsonl not found in dataset inputs - Stage 2 will generate it.")
else:
    print(f"mathtraincot.jsonl already in working dir "
          f"({os.path.getsize(_cot_dst)/1e6:.1f} MB) - Stage 2 will skip.")

run_pipeline()


  Device  : cuda
  GPU     : NVIDIA H100 80GB HBM3
  Stages  : [1, 2, 3, 4, 5, 6, 7]
  Ratios  : [0.5, 0.6, 0.7, 0.8, 0.9]
  Model   : Qwen/Qwen2.5-7B-Instruct
  OutDir  : /kaggle/working
mathtraincot.jsonl not found in dataset inputs - Stage 2 will generate it.
  [21:36:21] kgout: credential -> /kaggle/input/datasets/gauravvybhavkabhai/kgout-token/kgout_token.json
[kgout 21:36:21] 🔑 Using OAuth2 user credentials
[kgout 21:36:21] ☁️  Google Drive connected → folder 11IYp2WZOh5fuIXpY432Kz2usG4XorBAu
[kgout 21:36:21] 📸 Snapshot: 0 existing file(s) in '/kaggle/working'
[kgout 21:36:21] 👀 kgout watching '/kaggle/working' every 180s → gdrive
  [21:36:21] kgout started - syncing to Google Drive.

  LOADING MODEL & TOKENIZER
  [21:36:21] Loading tokenizer from Qwen/Qwen2.5-7B-Instruct ...


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

  [01:24:05]   batch 1/8  size=64  input_len=104
  [01:24:34]   batch 2/8  size=64  input_len=124
  [01:25:07]   batch 3/8  size=64  input_len=126
  [01:25:40]   batch 4/8  size=64  input_len=153
  [01:26:14]   batch 5/8  size=64  input_len=156
[kgout 01:26:47] 📦 [MODIFIED] mathcheckpoint.json
[kgout 01:26:48]    ↳ Deleted old version: mathcheckpoint.json
  [01:26:49]   batch 6/8  size=64  input_len=194
[kgout 01:26:49]    ↳ Uploaded to GDrive: mathcheckpoint.json (id: 1phU16tpZbeaTBfsqK3Kw_9k4Cmi350oA)
  [01:27:24]   batch 7/8  size=64  input_len=254
  [01:28:02]   batch 8/8  size=52  input_len=859
  [01:28:53] evaluate_batched DONE -> Acc=60.8%  AvgTok=489.86  elapsed=287.8s
    -> checkpoint saved (30 methods)
  [01:28:53] [LoRASoft0.9]  Acc=60.8%  AvgTok=489.86

    5e . Adaptive Compression Ratio (extension)
  [01:28:53] Evaluating LoRAAdaptive with per-problem γ based on difficulty ...
  [01:28:53]   Level→γ mapping: {'Level 1': 0.5, 1: 0.5, '1': 0.5, 'Level 2': 0.6, 2: 0.6, '2':

LoRAAdaptive:   0%|          | 0/8 [00:00<?, ?batch/s]

  [01:28:53]   batch 1/8  size=64  input_len=87
  [01:29:19]   batch 2/8  size=64  input_len=107
[kgout 01:29:49] 📦 [MODIFIED] mathcheckpoint.json
[kgout 01:29:50]    ↳ Deleted old version: mathcheckpoint.json
[kgout 01:29:51]    ↳ Uploaded to GDrive: mathcheckpoint.json (id: 14Lo9Xp8C7uVDgMkVQehjGAaZZVpTpZGQ)
  [01:29:52]   batch 3/8  size=64  input_len=109
  [01:30:24]   batch 4/8  size=64  input_len=136
  [01:30:57]   batch 5/8  size=64  input_len=139
  [01:31:31]   batch 6/8  size=64  input_len=177
  [01:32:06]   batch 7/8  size=64  input_len=237
  [01:32:43]   batch 8/8  size=52  input_len=842
  [01:33:34] evaluate_batched DONE -> Acc=60.2%  AvgTok=483.09  elapsed=281.1s
    -> checkpoint saved (31 methods)
  [01:33:34] [LoRAAdaptive]  Acc=60.2%  AvgTok=483.09
  [01:33:34]   Adaptive per-level breakdown:
  [01:33:34]     1 (γ=0.5): Acc=79.1%  AvgTok=203.3  n=43
  [01:33:34]     2 (γ=0.6): Acc=72.2%  AvgTok=324.8  n=90
  [01:33:34]     3 (γ=0.7): Acc=70.5%  AvgTok=417.7  n=105
  [0

/tmp/ipykernel_106/3039475281.py:475: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  sns.boxplot(data=dist_df, x="Method", y="Tokens", palette="Blues", ax=ax)


  [01:33:36] [fig] saved -> /kaggle/working/math_fig3_token_distribution.png (227 KB)
  [01:33:36] [fig] saved -> /kaggle/working/math_fig4_accuracy_drop_vs_savings.png (194 KB)
  [01:33:37] [fig] saved -> /kaggle/working/math_fig5_grouped_by_ratio.png (183 KB)
  [01:33:37] [fig] saved -> /kaggle/working/math_fig6_lora_triplet.png (142 KB)
  [01:33:38] [fig] saved -> /kaggle/working/math_fig7_all_methods_bar.png (429 KB)
  [01:33:39] [fig] saved -> /kaggle/working/math_fig8_subject_accuracy.png (886 KB)
  [01:33:39] All 8 figures complete.

  STAGE 7 . Zipping all outputs
  [01:33:39]   added to ZIP: math_fig1_truncation_analysis.png
  [01:33:39]   added to ZIP: math_fig2_method_heatmap.png
  [01:33:39]   added to ZIP: math_fig3_token_distribution.png
  [01:33:39]   added to ZIP: math_fig4_accuracy_drop_vs_savings.png
  [01:33:39]   added to ZIP: math_fig5_grouped_by_ratio.png
  [01:33:39]   added to ZIP: math_fig6_lora_triplet.png
  [01:33:39]   added to ZIP: math_fig7_all_methods_bar